# PhotoShield — E4 (real editor) and E5 (transfer)

Design §10. This notebook runs the two experiments that need a GPU and model downloads, which the
cloud container cannot do (CLAUDE.md A15). The logic is in `photoshield/experiments/edit_test.py`
and is unit-tested there. This notebook only drives it.

**Runtime → Change runtime type → GPU** (a free T4 is enough). With the defaults (6 photos ×
5 prompts × 5 seeds × 3 edits) E4 takes roughly 15–25 minutes on a T4; E5 a few minutes.

**What you get back:** `photoshield_results_e4_e5.zip`, holding `photoshield/results/e4/…` and `e5/…`.
Unzip it into your clone's `photoshield/results/` and run

    python -m photoshield.analysis.make_tables

Every number in the report's E4/E5 tables comes from those files. Never type a number into a table.

**Verdict rule:** "the cloak degrades edits" is claimed only where the bootstrap 95% CI of the
paired difference (cloaked vs original, and cloaked vs the same-strength random-noise control)
excludes 0. If it doesn't, the table says so.

## 1. Get the code

In [ ]:
# If the repository is private, upload a zip of your clone instead and unzip it to /content/BlockVerify.
import os
REPO_URL = "https://github.com/Shubhangam-Singh/BlockVerify.git"
BRANCH = "ss2005/modest-fermat-hflkcl"
if not os.path.exists("/content/BlockVerify"):
    !git clone -q -b {BRANCH} {REPO_URL} /content/BlockVerify
%cd /content/BlockVerify
!git log -1 --format='%h %s'

In [ ]:
!pip install -q diffusers transformers accelerate lpips scikit-image pynacl

## 2. Determinism

Seeds are fixed and recorded. On GPU some kernels have no deterministic implementation, so `warn_only=True`; the exact versions and device go into `config.json`.

In [ ]:
import os, random, sys, json, platform
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import numpy as np, torch
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cudnn.benchmark = False
device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "Switch the runtime to GPU first."
import diffusers, transformers
VERSIONS = {"torch": torch.__version__, "diffusers": diffusers.__version__,
            "transformers": transformers.__version__, "gpu": torch.cuda.get_device_name(0),
            "python": platform.python_version()}
print(VERSIONS)

## 3. Photos

Default: scikit-image's bundled public-domain photos. To use your own, upload them in the next cell (they stay in this Colab session; only metrics and grids are written).

In [ ]:
from photoshield.experiments.edit_test import load_images, PROMPTS
USE_OWN_PHOTOS = False      # True: upload JPEG/PNG files when prompted
N_IMAGES = 6
if USE_OWN_PHOTOS:
    from google.colab import files
    os.makedirs("/content/my_photos", exist_ok=True)
    for name, data in files.upload().items():
        open(os.path.join("/content/my_photos", name), "wb").write(data)
    images = load_images("/content/my_photos", N_IMAGES)
    IMAGE_SET = "own photos"
else:
    images = load_images("skimage", N_IMAGES)
    IMAGE_SET = "skimage"
print(IMAGE_SET, [n for n, _ in images])
print("prompts:", PROMPTS)

## 4. E4 — does InstructPix2Pix degrade on cloaked photos?

The cloak is crafted against the SD VAE (`stabilityai/sd-vae-ft-mse`) with EOT (resize, blur, JPEG). Each photo is edited three ways with the same prompt and seed: original, cloaked, and random ±ε noise (the control).

In [ ]:
from photoshield.cloak.encoders import load_encoder
from photoshield.experiments import edit_test as E

EPS, STEPS, SIZE = 8, 40, 512
EDIT_SEEDS = (0, 1, 2, 3, 4)

enc = load_encoder("sdvae").to(device)
editor, editor_info = E.instruct_pix2pix(device=device, steps=20)
editor_info.update(VERSIONS, image_set=IMAGE_SET)
lp = E.lpips_scorer(device)
cd = E.clip_directional(device)

e4_dir = E.run_e4(images, enc, editor, lp, cd, eps=EPS, steps=STEPS, size=SIZE,
                  prompts=PROMPTS, edit_seeds=EDIT_SEEDS, seed=SEED, device=device,
                  editor_info=editor_info, encoder_name="sdvae")
print("written:", e4_dir)

In [ ]:
# Side-by-side grids (first edit seed): original | edit(original) | cloaked | edit(cloaked) | edit(noise)
from IPython.display import Image as IPImage, display
for f in sorted(os.listdir(os.path.join(e4_dir, "grids")))[:3]:
    print(f); display(IPImage(os.path.join(e4_dir, "grids", f), width=900))

## 5. E5 — does an sdvae cloak transfer?

The cloak is crafted on the SD VAE only, then measured on a different VAE (SDXL's) and on ResNet-50 features, against the same-strength random-noise control.

In [ ]:
del editor; torch.cuda.empty_cache()
targets = {"sdxlvae": load_encoder("sdxlvae").to(device), "resnet50": load_encoder("resnet50").to(device)}
e5_dir = E.run_e5(images, enc, targets, eps=EPS, steps=STEPS, size=256, seed=SEED, device=device,
                  source_name="sdvae")
print("written:", e5_dir)

## 6. Tables and download

In [ ]:
from photoshield.analysis import make_tables
print(make_tables.render("photoshield/results"))

In [ ]:
import shutil
shutil.make_archive("/content/photoshield_results_e4_e5", "zip", "photoshield/results", ".")
from google.colab import files
files.download("/content/photoshield_results_e4_e5.zip")

## Reading the results honestly

- A positive LPIPS difference with a CI above 0 means the cloak changed InstructPix2Pix's edit
  *more than random noise of the same strength*. It does not mean the photo is "protected".
- InstructPix2Pix starts from the same SD VAE the cloak targets. Closed editors (Gemini, Grok,
  Midjourney) are not tested here, and nothing here suggests the cloak affects them.
- E5 tells you how far the effect carries to encoders the cloak never saw. A CI that includes 0
  means no transfer was shown.
- The provenance layer (E6/E7) does not depend on any of this: a registered photo stays
  verifiable whether or not the cloak survives.